# ALOHA Robot — ACT Training & Evaluation (Colab)

Train/evaluate an ALOHA policy in the same practical Colab style as the quadruped workflow.

## Model selected (found on Hugging Face)
- **`lerobot/act_aloha_sim_transfer_cube_human`**
- Task: **`AlohaTransferCube-v0`**
- Framework: **LeRobot + gym-aloha**

This notebook does 3 things:
1. Installs LeRobot with ALOHA sim support
2. Evaluates the pretrained HF model
3. (Optional) Fine-tunes ACT further on ALOHA dataset


## 1) Runtime setup

Use **GPU runtime** in Colab (`Runtime -> Change runtime type -> GPU`).


In [ ]:
!nvidia-smi

In [ ]:
# Clone LeRobot at the commit used by the model card for compatibility
!rm -rf /content/lerobot
!git clone https://github.com/huggingface/lerobot.git /content/lerobot
%cd /content/lerobot
!git checkout 3c0a209

In [ ]:
# Install LeRobot + ALOHA simulation extras
!pip -q install -U pip
!pip -q install -e ".[aloha]"

# Optional experiment tracking
!pip -q install wandb

In [ ]:
# Quick sanity: imports
import torch
import gymnasium as gym
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2) Evaluate pretrained ALOHA model from Hugging Face

We will use:
- `policy.path = lerobot/act_aloha_sim_transfer_cube_human`
- `env.task = AlohaTransferCube-v0`


In [ ]:
# (Optional) login if you hit hub rate limits/private assets
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# Evaluate pretrained policy (quick pass)
!python lerobot/scripts/eval.py   --policy.path=lerobot/act_aloha_sim_transfer_cube_human   --output_dir=outputs/eval/act_aloha_transfer_cube_hf   --env.type=aloha   --env.task=AlohaTransferCube-v0   --eval.n_episodes=100   --eval.batch_size=20   --device=cuda   --use_amp=false

In [ ]:
# Inspect evaluation outputs
import json, pathlib
base = pathlib.Path('outputs/eval/act_aloha_transfer_cube_hf')
print('Eval dir exists:', base.exists())

for p in sorted(base.rglob('*')):
    if p.is_file() and p.suffix in ['.json', '.txt', '.md']:
        print('-', p)

eval_json = None
for cand in base.rglob('eval_info.json'):
    eval_json = cand
    break

if eval_json:
    print()
    print('Found eval_info.json:', eval_json)
    data = json.loads(eval_json.read_text())
    # Print a compact preview
    if isinstance(data, dict):
        for k in list(data.keys())[:12]:
            print(k, '=>', str(data[k])[:200])
else:
    print('No eval_info.json found. Check command logs above.')

## 3) Optional fine-tuning (same ACT format)

This follows the model card training style, using ALOHA transfer-cube dataset.


In [ ]:
# Optional: W&B offline to avoid login prompt
!wandb offline

In [ ]:
# Fine-tune ACT on ALOHA transfer cube (adjust steps/time as needed)
!python lerobot/scripts/train.py   --output_dir=outputs/train/act_aloha_transfer_ft   --policy.type=act   --dataset.repo_id=lerobot/aloha_sim_transfer_cube_human   --env.type=aloha   --env.task=AlohaTransferCube-v0   --wandb.enable=false

In [ ]:
# Evaluate the fine-tuned checkpoint (update checkpoint path if needed)
!python lerobot/scripts/eval.py   --policy.path=outputs/train/act_aloha_transfer_ft/checkpoints/080000/pretrained_model   --output_dir=outputs/eval/act_aloha_transfer_ft_080000   --env.type=aloha   --env.task=AlohaTransferCube-v0   --eval.n_episodes=200   --eval.batch_size=20   --device=cuda   --use_amp=false

## 4) Compare pretrained vs fine-tuned


In [ ]:
import json, pathlib, pandas as pd

def load_success(eval_dir):
    eval_dir = pathlib.Path(eval_dir)
    f = None
    for cand in eval_dir.rglob('eval_info.json'):
        f = cand
        break
    if f is None:
        return None
    data = json.loads(f.read_text())

    # Try common keys
    for k in ['success_rate', 'eval/success_rate', 'metrics/success_rate']:
        if isinstance(data, dict) and k in data:
            return float(data[k])

    # Fallback: infer from per-episode entries
    if isinstance(data, dict):
        for k, v in data.items():
            if isinstance(v, list) and len(v) and isinstance(v[0], dict):
                if 'success' in v[0]:
                    vals = [int(x.get('success', 0)) for x in v]
                    return sum(vals) / max(1, len(vals))
    return None

rows = [
    ['HF pretrained', load_success('outputs/eval/act_aloha_transfer_cube_hf')],
    ['Fine-tuned', load_success('outputs/eval/act_aloha_transfer_ft_080000')],
]

df = pd.DataFrame(rows, columns=['run', 'success_rate'])
display(df)

## 5) Notes

- If the fine-tune checkpoint path differs, list files under `outputs/train/act_aloha_transfer_ft/checkpoints/` and update the eval cell.
- For longer training, increase runtime and monitor disk usage.
- For production demos, render/save rollout videos from eval outputs.
